
# Linear and Polynomial Regression

This notebook is intentionally self-contained. It does not import any local utility module, so you can open this file alone during study and see the full statistical workflow, simulation setup, model fitting, evaluation, plotting, and interpretation pattern in one place.



## What problem are we solving?

Use linear regression when the target is numeric and the mean response is approximately linear in the predictors. Use polynomial terms when a one-dimensional predictor has visible curvature.

**Highest-probability exam pattern:** Choose polynomial degree and explain the bias-variance tradeoff using train/test MSE or cross-validation.



## Assumptions, intuition, and theory

A low-degree model can underfit because it has high bias. A high-degree polynomial can overfit because it has high variance. Test MSE or CV MSE is the quantity to minimize when the goal is prediction.



    ## Python method notes and documentation

    `PolynomialFeatures` creates powers of x, `make_pipeline` applies the same transformation inside fitting and prediction, `LinearRegression.fit` estimates coefficients, and `predict` computes fitted responses.

    Documentation links:

    - [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)
- [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)


## Fully self-contained worked example

Every code line below is commented so the workflow can be scanned quickly under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for model-comparison results.
import matplotlib.pyplot as plt  # Import plotting tools for fitted curves and error curves.
from sklearn.base import clone  # Import clone so each fit uses a fresh estimator.
from sklearn.linear_model import LinearRegression  # Import ordinary least-squares regression.
from sklearn.metrics import mean_squared_error  # Import the explicit regression error metric.
from sklearn.model_selection import train_test_split  # Import train/test splitting.
from sklearn.pipeline import make_pipeline  # Import pipeline construction for basis expansion plus regression.
from sklearn.preprocessing import PolynomialFeatures  # Import polynomial feature expansion.
RANDOM_SEED = 123  # Store the reproducibility seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_sine_regression_data(n=120, noise=0.35, random_state=123):  # Define a reusable nonlinear regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random number generator.
    x = np.sort(rng.uniform(0.0, 3.0, size=n))  # Draw predictor values and sort them for cleaner plots.
    signal = 4.0 + np.sin(3.0 * x)  # Define the true nonlinear mean function.
    y = signal + rng.normal(0.0, noise, size=n)  # Add Gaussian noise to create observed responses.
    X = x.reshape(-1, 1)  # Convert the predictor to a two-dimensional sklearn design matrix.
    return X, y, signal  # Return predictors, observed response, and noiseless truth.

def make_polynomial_regression_data(n=140, noise=2.0, random_state=123):  # Define a curved polynomial regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random number generator.
    x = rng.uniform(-2.5, 2.5, size=n)  # Draw one numeric predictor over a moderate range.
    signal = 1.0 - 2.0 * x + 0.8 * x**2 + 0.7 * x**3  # Define the true polynomial mean curve.
    y = signal + rng.normal(0.0, noise, size=n)  # Add independent Gaussian errors.
    X = x.reshape(-1, 1)  # Reshape the predictor for sklearn estimators.
    return X, y, signal  # Return the design matrix, response, and true mean.

def train_test_mse(model, X, y, test_size=0.30, random_state=123):  # Define a local train/test regression evaluator.
    X_train, X_test, y_train, y_test = train_test_split(  # Split observations into training and testing parts.
        X,  # Pass the predictor matrix to the splitter.
        y,  # Pass the numeric response vector to the splitter.
        test_size=test_size,  # Hold out this fraction for honest testing.
        random_state=random_state,  # Fix the random split for reproducibility.
    )  # Finish the train/test split call.
    fitted_model = clone(model)  # Clone the estimator so repeated calls do not reuse fitted state.
    fitted_model.fit(X_train, y_train)  # Fit the model only on the training data.
    train_pred = fitted_model.predict(X_train)  # Predict the training responses to assess in-sample error.
    test_pred = fitted_model.predict(X_test)  # Predict the held-out responses to assess future error.
    train_mse = mean_squared_error(y_train, train_pred)  # Compute training mean squared error.
    test_mse = mean_squared_error(y_test, test_pred)  # Compute test mean squared error.
    return fitted_model, train_mse, test_mse  # Return the fitted model and both error estimates.


In [ ]:
X, y, true_signal = make_polynomial_regression_data(n=140, noise=2.0, random_state=RANDOM_SEED)  # Simulate a nonlinear numeric-response data set.
rows = []  # Create an empty list for train/test MSE results.
for degree in range(1, 9):  # Try polynomial degrees from simple to flexible.
    model = make_pipeline(PolynomialFeatures(degree=degree, include_bias=False), LinearRegression())  # Build polynomial regression for this degree.
    fitted_model, train_mse, test_mse = train_test_mse(model, X, y, random_state=RANDOM_SEED)  # Fit and evaluate the candidate model.
    rows.append({'degree': degree, 'train_mse': train_mse, 'test_mse': test_mse})  # Store both in-sample and held-out errors.
results = pd.DataFrame(rows)  # Convert the accumulated rows to a table.
best_degree = int(results.loc[results['test_mse'].idxmin(), 'degree'])  # Select the degree with smallest test MSE.
best_model = make_pipeline(PolynomialFeatures(degree=best_degree, include_bias=False), LinearRegression())  # Recreate the selected model.
best_model.fit(X, y)  # Fit the selected model on the full simulated data for plotting.
x_grid = np.linspace(X[:, 0].min(), X[:, 0].max(), 200).reshape(-1, 1)  # Create a smooth plotting grid.
y_grid = best_model.predict(x_grid)  # Predict the fitted curve over the grid.
display(results)  # Display the degree-comparison table.
print(f'Best degree by test MSE: {best_degree}')  # Print the selected degree.
plt.figure(figsize=(6, 4))  # Create a figure for the fitted polynomial curve.
plt.scatter(X[:, 0], y, s=22, alpha=0.6, label='observed data')  # Plot observed points.
plt.plot(x_grid[:, 0], y_grid, color='red', label=f'degree {best_degree} fit')  # Plot the selected fitted curve.
plt.xlabel('x')  # Label the predictor axis.
plt.ylabel('y')  # Label the response axis.
plt.title('Polynomial regression selected by test MSE')  # Title the plot.
plt.legend()  # Show plot labels.
plt.show()  # Render the fitted-curve plot.
plt.figure(figsize=(6, 4))  # Create a second figure for overfitting diagnostics.
plt.plot(results['degree'], results['train_mse'], marker='o', label='train MSE')  # Plot training error by degree.
plt.plot(results['degree'], results['test_mse'], marker='o', label='test MSE')  # Plot testing error by degree.
plt.xlabel('polynomial degree')  # Label the tuning axis.
plt.ylabel('MSE')  # Label the error axis.
plt.title('Training error can fall while test error rises')  # State the main interpretation in the title.
plt.legend()  # Show the train/test labels.
plt.show()  # Render the error plot.



## Exam-style problem prompt

Simulate or receive one numeric predictor and one numeric response. Fit polynomial regressions of several degrees, estimate test MSE, choose the best degree, and explain whether overfitting appears.



    ## Adaptable code pattern

    ```python
    # Step 1: Simulate or load numeric X and y.
# Step 2: Loop over candidate polynomial degrees.
# Step 3: For each degree, fit PolynomialFeatures + LinearRegression on training data.
# Step 4: Predict held-out data and compute MSE.
# Step 5: Choose the degree with smallest test or CV MSE.
# Step 6: Plot fitted curve and MSE curve, then explain bias-variance tradeoff.
    ```



## What to remember for the exam

Polynomial regression is still linear regression after transforming features. The exam usually cares less about coefficient interpretation and more about prediction error, degree choice, and overfitting.
